# Linear recovery and identifiability

## Checkpoint A — controlled noiseless OLS

This executable checkpoint uses one seeded generated world, not a corpus survey. The production symbolic graph generates mixed nontrivial treatment trajectories and the observed outcome. The fit uses the true mechanism basis: each treatment column is carried over first and saturated second with its parameter-only generated scale.

Assumptions for this checkpoint:

- all three direct treatment inputs and both observed controls are active;
- geometric, Weibull, and geometric carryover are paired with Hill, Michaelis–Menten, and root saturation;
- the latent node is structurally disconnected, with no treatment mediation or treatment interactions;
- the intrinsic baseline is exactly constant at 5, the floor is disabled, and outcome noise is exactly zero;
- four generated warmup rows are retained as initial carryover history and excluded from the fit;
- OLS is run on the raw `[mechanism features, controls, intercept]` design, without centering or rescaling.

This is a positive identifiability construction. It does not estimate the frequency of rank deficiency or address noisy recovery (Checkpoint B) or an intentionally non-identifiable likelihood (Checkpoint C).


In [1]:
from tests.test_linear_recovery import (
    N_TIME_STEPS,
    SEED,
    WARMUP,
    _controlled_world,
    _fit,
    _mechanism_features,
)
import numpy as np

world = _controlled_world()
coefficient, truth, rank, condition = _fit(world)
features_full = _mechanism_features(
    world.outputs["treatments"], world.outputs["saturation_scale"], world.params
)
window = slice(WARMUP, None)
design = np.column_stack([
    features_full[window],
    world.outputs["covariates"][window],
    np.ones(N_TIME_STEPS),
])
error = coefficient - truth
print(f"seed={SEED}, generated_rows={N_TIME_STEPS + WARMUP}, warmup={WARMUP}")
print(f"raw_design_shape={design.shape}, rank={rank}/{design.shape[1]}")
print(f"raw_condition={condition:.12g}, max_abs_error={np.max(np.abs(error)):.12g}")
print("column        truth                estimate             error")
for name, target, estimate, delta in zip(
    ["treatment_0", "treatment_1", "treatment_2", "control_0", "control_1", "intercept"],
    truth,
    coefficient,
    error,
    strict=True,
):
    print(f"{name:12s} {target: .16g} {estimate: .16g} {delta: .3g}")


/home/teemu/pymc-labs/prior-generator/.worktrees/issue-30-identifiable-linear-recovery/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


seed=20261005, generated_rows=68, warmup=4
raw_design_shape=(64, 6), rank=6/6
raw_condition=71.8975337829, max_abs_error=7.43849426499e-15
column        truth                estimate             error
treatment_0   1.15  1.15  0
treatment_1   0.82  0.8200000000000074  7.44e-15
treatment_2   1.08  1.080000000000001  4.44e-16
control_0     0.35  0.3500000000000037  3.72e-15
control_1    -0.28 -0.2800000000000023 -2.28e-15
intercept     5  4.999999999999998 -1.78e-15


### Independent feature and history checks

The generated direct contributions agree with the independently re-applied mechanism features multiplied by the generator's true treatment coefficients. Recomputing carryover from only the post-warmup window changes the first retained feature, demonstrating why the generated history is part of the basis.


In [2]:
np.testing.assert_allclose(
    world.outputs["contributions_observed"][window],
    features_full[window] * np.asarray(world.params["beta"]),
    rtol=2e-12,
    atol=2e-12,
)
reset_features = _mechanism_features(
    world.outputs["treatments"][window], world.outputs["saturation_scale"], world.params
)
history_difference = float(np.max(np.abs(reset_features[0] - features_full[WARMUP])))
np.testing.assert_array_equal(world.outputs["outcome_noise"], 0.0)
np.testing.assert_array_equal(world.outputs["baseline_intrinsic"], 5.0)
np.testing.assert_allclose(
    world.outputs["baseline"],
    world.outputs["baseline_intrinsic"] + world.outputs["covariate_contribution"].sum(axis=1),
    rtol=0.0,
    atol=2e-15,
)
assert rank == design.shape[1]
assert condition < 100.0
np.testing.assert_allclose(coefficient, truth, rtol=2e-12, atol=2e-12)
print(f"max feature contribution discrepancy={np.max(np.abs(world.outputs['contributions_observed'][window] - features_full[window] * world.params['beta'])):.3g}")
print(f"reset-at-window history discrepancy={history_difference:.6g}")


max feature contribution discrepancy=1.11e-16
reset-at-window history discrepancy=0.585202


### Result and limitation

The raw design is full rank and its condition number is reported above in the original feature units. The coefficient errors are at floating-point roundoff. The condition number is a property of this deliberately controlled trajectory, not a claim about the prior or general generated worlds. A future checkpoint must separately test noise and construct an explicit non-identifiable likelihood; neither is included here.
